# Disque 100 latent typologies with DAMICORE

Each report is one DAMICORE object. PostgreSQL only supplies the data; DAMICORE is the only clustering method. The default sample has 500 reports because pairwise distances do not scale to all 371,117 reports at once.

In [12]:
%pip install -q "damicore==0.2.0" "psycopg[binary]>=3.2,<4"

Note: you may need to restart the kernel to use updated packages.


In [14]:
import os
from pathlib import Path

import pandas as pd
import psycopg
from psycopg import sql
from damicore import estimate, run

DATABASE_URL = os.getenv("DISQUE100_DATABASE_URL", "postgresql:///disque100")
SAMPLE_SIZE = 1000
SAMPLE_SEED = "42"
START_DATE = "2026-01-01"
END_DATE = "2026-07-01"

RUN_NAME = "disque100-damicore-20260821-03"
CORPUS_DIR = Path("output") / f"{RUN_NAME}-corpus"
RUN_DIR = Path("output") / f"{RUN_NAME}-run"
MEMBERSHIP_FILE = Path("output") / f"{RUN_NAME}-membership.csv"

for path in (CORPUS_DIR, RUN_DIR, MEMBERSHIP_FILE):
    if path.exists():
        raise FileExistsError(f"Choose a new RUN_NAME; output already exists: {path}")

CORPUS_DIR.mkdir(parents=True)

In [15]:
# Build one text file per report from the PostgreSQL table.
query = sql.SQL("""
    WITH cohort AS (
        SELECT source_hash
        FROM {table}
        WHERE registered_at >= %s AND registered_at < %s
        GROUP BY source_hash
        ORDER BY md5(source_hash || %s)
        LIMIT %s
    )
    SELECT
        reports.source_hash,
        jsonb_agg(
            jsonb_strip_nulls(
                to_jsonb(reports) - ARRAY['id', 'source_hash', 'registered_at']::text[]
            )
            ORDER BY reports.id
        )::text AS report_text
    FROM {table} AS reports
    JOIN cohort USING (source_hash)
    GROUP BY reports.source_hash
    ORDER BY reports.source_hash
""").format(table=sql.Identifier("public", "disque100_reports"))

with psycopg.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query, (START_DATE, END_DATE, SAMPLE_SEED, SAMPLE_SIZE))
        reports = cursor.fetchall()

if len(reports) < 2:
    raise RuntimeError("DAMICORE needs at least two reports.")

for source_hash, report_text in reports:
    (CORPUS_DIR / f"{source_hash}.txt").write_text(report_text, encoding="utf-8")

print(f"Created {len(reports)} report objects in {CORPUS_DIR}")

Created 1000 report objects in output/disque100-damicore-20260821-03-corpus


In [16]:
preview = estimate(CORPUS_DIR, source_kind="files")
display(preview.model_dump())

if not preview.within_limits:
    raise RuntimeError(f"DAMICORE limits exceeded: {preview.violations}")

{'source_kind': 'files',
 'source_paths': (PosixPath('/Users/erickpatrickbarcelos/codes/research/output/disque100-damicore-20260821-03-corpus/0012F1545B613D6D00730B2F524D915BE2AAB2587F0A4A7F5563A84C55CEA2C1.txt'),
  PosixPath('/Users/erickpatrickbarcelos/codes/research/output/disque100-damicore-20260821-03-corpus/00515441524105EA7CD870DC5415F64B650A09C43486D93AE5560C056BCCA129.txt'),
  PosixPath('/Users/erickpatrickbarcelos/codes/research/output/disque100-damicore-20260821-03-corpus/005C36B5855B7FB8845FD5A8712E2222933969412F9D5346CBAEDCB9583E7ACC.txt'),
  PosixPath('/Users/erickpatrickbarcelos/codes/research/output/disque100-damicore-20260821-03-corpus/0094AE3ED0B2C3482D168E00D319D932E4B979C8FC31F88E887A8176C1AC3139.txt'),
  PosixPath('/Users/erickpatrickbarcelos/codes/research/output/disque100-damicore-20260821-03-corpus/00957521478D60F72D40F2D607D9AD695D6F7B08EC2F83D643AAC3A3701E0577.txt'),
  PosixPath('/Users/erickpatrickbarcelos/codes/research/output/disque100-damicore-20260821-03-

In [ ]:
result = run(CORPUS_DIR, source_kind="files", output_dir=RUN_DIR)

try:
    membership = result.membership.copy()
    membership["source_hash"] = membership["label"].str.removesuffix(".txt")
    membership.to_csv(MEMBERSHIP_FILE, index=False)

    display(membership.head(20))
    display(membership.groupby("cluster").size().rename("report_count"))
    display(result.distance_matrix.head(10))
    print(result.tree_newick[:1000])
finally:
    result.close()

print(f"Membership saved to {MEMBERSHIP_FILE}")

distance: 0pair [00:00, ?pair/s]